# Bybit Loader 실습

이 노트북은 프로젝트의 `DataLoaderStrategy` 방식과 유사한 Loader 구조로 Bybit 공개 시세 API에서 캔들 및 티커 데이터를 가져옵니다.

공개 시장 데이터만 조회하므로 API 키 없이 실행할 수 있습니다.

In [1]:
# 처음 한 번만 실행하면 됩니다.
# %pip install requests pandas python-dotenv pyarrow pytest

import hashlib
import hmac
import importlib
import os
import time
from abc import ABC, abstractmethod
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import pandas as pd
import requests

_dotenv = importlib.util.find_spec("dotenv")
if _dotenv:
    load_dotenv = importlib.import_module("dotenv").load_dotenv
else:
    def load_dotenv():
        return False

load_dotenv()
WORKSPACE = Path.cwd()
DATA_DIR = WORKSPACE / "data"
DATA_DIR.mkdir(exist_ok=True)
print(f"작업 폴더: {WORKSPACE}")

작업 폴더: c:\Users\baram\dongseop\dongseop


In [2]:
@dataclass
class BybitConfig:
    base_url: str = os.getenv("BYBIT_BASE_URL", "https://api.bybit.com")
    api_key: Optional[str] = os.getenv("BYBIT_API_KEY")
    api_secret: Optional[str] = os.getenv("BYBIT_API_SECRET")
    recv_window: int = 5000

    def validate(self, require_auth: bool = False) -> None:
        if not self.base_url.startswith(("http://", "https://")):
            raise ValueError("BYBIT_BASE_URL은 http 또는 https URL이어야 합니다.")
        if require_auth and not (self.api_key and self.api_secret):
            raise ValueError("인증 요청에는 API key와 secret이 필요합니다.")
        if bool(self.api_key) != bool(self.api_secret):
            raise ValueError("BYBIT_API_KEY와 BYBIT_API_SECRET은 함께 설정해야 합니다.")

config = BybitConfig()
config.validate()
print(f"endpoint={config.base_url}, public_mode={not bool(config.api_key and config.api_secret)}")

endpoint=https://api.bybit.com, public_mode=True


In [3]:
class Loader(ABC):
    @abstractmethod
    def load(self) -> pd.DataFrame:
        raise NotImplementedError

    @abstractmethod
    def validate(self, data: Any) -> None:
        raise NotImplementedError

    @abstractmethod
    def transform(self, data: Any) -> pd.DataFrame:
        raise NotImplementedError


class BybitAPIError(RuntimeError):
    pass

In [4]:
class BybitLoader(Loader):
    columns = ["timestamp", "open", "high", "low", "close", "volume", "turnover"]

    def __init__(self, config: BybitConfig, symbol="BTCUSDT", interval="15", category="linear", count=200, max_retries=3):
        self.config = config
        self.config.validate()
        self.symbol = symbol.upper()
        self.interval = interval
        self.category = category
        self.count = count
        self.max_retries = max_retries
        self.session = requests.Session()
        self.session.headers.update({"Content-Type": "application/json"})

    def _auth_headers(self, params: dict[str, Any]) -> dict[str, str]:
        if not (self.config.api_key and self.config.api_secret):
            return {}
        timestamp = str(int(time.time() * 1000))
        query = "&".join(f"{key}={params[key]}" for key in sorted(params))
        payload = f"{timestamp}{self.config.api_key}{self.config.recv_window}{query}"
        signature = hmac.new(self.config.api_secret.encode(), payload.encode(), hashlib.sha256).hexdigest()
        return {"X-BAPI-API-KEY": self.config.api_key, "X-BAPI-TIMESTAMP": timestamp, "X-BAPI-RECV-WINDOW": str(self.config.recv_window), "X-BAPI-SIGN": signature, "X-BAPI-SIGN-TYPE": "2"}

    def _request(self, endpoint: str, params=None, authenticated=False):
        params = params or {}
        headers = self._auth_headers(params) if authenticated else {}
        for attempt in range(self.max_retries + 1):
            try:
                response = self.session.get(f"{self.config.base_url}{endpoint}", params=params, headers=headers, timeout=15)
                if response.status_code == 429 or response.status_code >= 500:
                    raise requests.HTTPError(f"HTTP {response.status_code}")
                response.raise_for_status()
                payload = response.json()
                if payload.get("retCode") != 0:
                    raise BybitAPIError(f"Bybit API 오류 {payload.get('retCode')}: {payload.get('retMsg')}")
                return payload
            except requests.RequestException as error:
                if attempt >= self.max_retries:
                    raise BybitAPIError(f"요청 실패: {error}") from error
                time.sleep(2 ** attempt)
        raise BybitAPIError("요청 실패")

    def get_kline(self, symbol=None, interval=None, start=None, end=None, limit=None):
        requested_count = limit or self.count
        if requested_count < 1:
            raise ValueError("count/limit은 1 이상이어야 합니다.")
        rows = []
        cursor_end = int(end.timestamp() * 1000) if end else None
        start_ms = int(start.timestamp() * 1000) if start else None
        while len(rows) < requested_count:
            page_limit = min(1000, requested_count - len(rows))
            params = {"category": self.category, "symbol": (symbol or self.symbol).upper(), "interval": interval or self.interval, "limit": page_limit}
            if start_ms is not None:
                params["start"] = start_ms
            if cursor_end is not None:
                params["end"] = cursor_end
            payload = self._request("/v5/market/kline", params)
            page_rows = payload.get("result", {}).get("list", [])
            if not page_rows:
                break
            rows.extend(page_rows)
            oldest = min(int(row[0]) for row in page_rows)
            if start_ms is not None and oldest <= start_ms:
                break
            cursor_end = oldest - 1
            if len(page_rows) < page_limit:
                break
        return rows[:requested_count]

    def get_ticker(self, symbol=None):
        payload = self._request("/v5/market/tickers", {"category": self.category, "symbol": (symbol or self.symbol).upper()})
        tickers = payload.get("result", {}).get("list", [])
        if not tickers:
            raise BybitAPIError("티커 응답이 비어 있습니다.")
        return tickers[0]

    def validate(self, data: Any) -> None:
        if not isinstance(data, list):
            raise TypeError("Kline 응답은 list이어야 합니다.")
        if any(not isinstance(row, list) or len(row) < 7 for row in data):
            raise ValueError("Kline 행의 컬럼 수가 올바르지 않습니다.")

    def transform(self, data: list[list[str]]) -> pd.DataFrame:
        self.validate(data)
        df = pd.DataFrame(data, columns=self.columns)
        df["timestamp_kst"] = pd.to_datetime(pd.to_numeric(df["timestamp"]), unit="ms", utc=True).dt.tz_convert("Asia/Seoul")
        numeric_columns = ["open", "high", "low", "close", "volume", "turnover"]
        df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric)
        df = df.sort_values("timestamp_kst").drop_duplicates("timestamp_kst").reset_index(drop=True)
        return df[["timestamp_kst", "open", "low", "high", "close", "volume"]]

    def load(self) -> pd.DataFrame:
        return self.transform(self.get_kline(limit=self.count))

In [5]:
loader = BybitLoader(config, symbol="BTCUSDT", interval="15", category="linear", count=200)
df = loader.load()
print(f"로드 행 수: {len(df)}")
display(df.head())
display(df.tail())

로드 행 수: 200


,timestamp_kst,open,low,high,close,volume
0,2026-08-26 16:30:00+09:00,78898.4,78845.0,79020.7,78952.0,316.384
1,2026-08-26 16:45:00+09:00,78952.0,78873.2,79028.9,78884.1,330.340
2,2026-08-26 17:00:00+09:00,78884.1,78777.8,79097.6,78996.3,441.366
3,2026-08-26 17:15:00+09:00,78996.3,78865.6,79071.6,78941.0,149.224
4,2026-08-26 17:30:00+09:00,78941.0,78859.8,78991.9,78972.9,167.169


,timestamp_kst,open,low,high,close,volume
195,2026-08-28 17:15:00+09:00,79704.8,79704.8,79808.0,79752.0,201.131
196,2026-08-28 17:30:00+09:00,79752.0,79671.7,79789.9,79677.0,365.632
197,2026-08-28 17:45:00+09:00,79677.0,79426.4,79698.4,79481.5,1001.426
198,2026-08-28 18:00:00+09:00,79481.5,79200.0,79481.6,79352.0,1285.451
199,2026-08-28 18:15:00+09:00,79352.0,79338.2,79440.4,79390.1,142.709


In [ ]:
print("결측치:")
print(df.isna().sum())
print(f"중복 timestamp: {df['timestamp_kst'].duplicated().sum()}")
print(f"시간 오름차순: {df['timestamp_kst'].is_monotonic_increasing}")
print(f"최신 종가: {df.iloc[-1]['close']}")

ticker = loader.get_ticker()
print(f"현재가(lastPrice): {ticker['lastPrice']}")
print(f"24시간 변동률: {ticker.get('price24hPcnt')}")

In [ ]:
csv_path = DATA_DIR / "bybit_btcusdt_15m.csv"
parquet_path = DATA_DIR / "bybit_btcusdt_15m.parquet"
df.to_csv(csv_path, index=False)
print(f"CSV 저장: {csv_path}")

try:
    df.to_parquet(parquet_path, index=False)
    print(f"Parquet 저장: {parquet_path}")
except ImportError:
    parquet_path = None
    print("Parquet은 pyarrow 설치 후 저장할 수 있습니다.")

csv_df = pd.read_csv(csv_path, parse_dates=["timestamp_kst"])
assert len(csv_df) == len(df)
assert list(csv_df.columns) == list(df.columns)
assert csv_df["timestamp_kst"].is_monotonic_increasing
print(f"CSV 재로딩 검증 성공: {len(csv_df)}행")

if parquet_path:
    parquet_df = pd.read_parquet(parquet_path)
    assert len(parquet_df) == len(df)
    assert list(parquet_df.columns) == list(df.columns)
    assert parquet_df["timestamp_kst"].is_monotonic_increasing
    print(f"Parquet 재로딩 검증 성공: {len(parquet_df)}행")

In [ ]:
def test_validate_and_transform():
    fake_rows = [
        ["1700000060000", "101", "102", "100", "101.5", "10", "1000"],
        ["1700000000000", "99", "101", "98", "100", "12", "1200"],
    ]
    transformed = BybitLoader(config, count=2).transform(fake_rows)
    assert list(transformed.columns) == ["timestamp_kst", "open", "low", "high", "close", "volume"]
    assert len(transformed) == 2
    assert transformed["timestamp_kst"].is_monotonic_increasing
    assert transformed["close"].dtype.kind in "fi"


def test_validate_rejects_invalid_rows():
    try:
        BybitLoader(config).validate([["only", "two"]])
    except ValueError:
        return
    raise AssertionError("잘못된 Kline 행을 거부해야 합니다.")


def test_api_failure_is_wrapped():
    class FailedSession:
        def get(self, *args, **kwargs):
            del args, kwargs
            raise requests.ConnectionError("mock network failure")

    test_loader = BybitLoader(config, max_retries=0)
    test_loader.session = FailedSession()
    try:
        test_loader._request("/v5/market/kline", {"category": "linear"})
    except BybitAPIError as error:
        assert "요청 실패" in str(error)
    else:
        raise AssertionError("네트워크 오류가 BybitAPIError로 변환되어야 합니다.")


test_validate_and_transform()
test_validate_rejects_invalid_rows()
test_api_failure_is_wrapped()
print("mock 기반 Loader 테스트 성공")

## 사용 메모

- `category="linear"`은 USDT 무기한 선물이고, 현물은 `category="spot"`으로 바꿉니다.
- 공개 Kline/티커 API는 API 키 없이 호출됩니다. 주문, 잔고 등 private API를 사용할 때만 `.env`에 `BYBIT_API_KEY`, `BYBIT_API_SECRET`을 설정합니다.
- Bybit Kline 응답은 최신 봉부터 오므로 `transform()`에서 한국 시간으로 변환한 뒤 과거에서 최신 순서로 정렬합니다.
- 기본 실행 결과는 `data/bybit_btcusdt_15m.csv`와 `data/bybit_btcusdt_15m.parquet`에 저장됩니다.